In [1]:
!pip install music21

In [2]:
from music21 import stream, note, chord

print("music21 imported successfully!")
print("Ready for MIDI music processing.")

music21 imported successfully!
Ready for MIDI music processing.


In [4]:
import os
import urllib.request

os.makedirs("midi_dataset", exist_ok=True)

url = "https://storage.googleapis.com/magentadata/models/music_vae/colab2.mid"
file_path = "midi_dataset/sample.mid"

urllib.request.urlretrieve(url, file_path)

print("MIDI dataset downloaded successfully!")
print("File:", file_path)

HTTPError: HTTP Error 404: Not Found

In [5]:
!git clone https://github.com/craffel/pretty-midi.git

Cloning into 'pretty-midi'...
remote: Enumerating objects: 1341, done.
remote: Counting objects: 100% (188/188), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 1341 (delta 162), reused 118 (delta 118), pack-reused 1153 (from 1)
Receiving objects: 100% (1341/1341), 12.69 MiB | 22.52 MiB/s, done.
Resolving deltas: 100% (795/795), done.


In [6]:
import os

print("Checking dataset folder...")

for root, dirs, files in os.walk("pretty-midi"):
    midi_files = [f for f in files if f.endswith((".mid", ".midi"))]
    if midi_files:
        print("MIDI files found:", len(midi_files))
        print("Example:", midi_files[:5])
        break
else:
    print("No MIDI files found in this folder.")

Checking dataset folder...
MIDI files found: 1
Example: ['example.mid']


In [7]:
from music21 import converter, note, chord

midi_file = "pretty-midi/test/data/test.mid"

score = converter.parse(midi_file)

notes = []

for element in score.flatten().notes:
    if isinstance(element, note.Note):
        notes.append(str(element.pitch))
    elif isinstance(element, chord.Chord):
        notes.append('.'.join(str(n) for n in element.normalOrder))

print("MIDI file processed successfully!")
print("Total notes/chords:", len(notes))
print("First 20:", notes[:20])

FileNotFoundError: Cannot find file in pretty-midi/test/data/test.mid

In [9]:
import os

midi_files = []

for root, dirs, files in os.walk("pretty-midi"):
    for file in files:
        if file.endswith((".mid", ".midi")):
            midi_files.append(os.path.join(root, file))

print("MIDI files found:", len(midi_files))

for file in midi_files[:10]:
    print(file)

MIDI files found: 1
pretty-midi/example.mid


In [10]:
from music21 import converter, note, chord

midi_file = midi_files[0]
score = converter.parse(midi_file)

notes = []

for element in score.flatten().notes:
    if isinstance(element, note.Note):
        notes.append(str(element.pitch))
    elif isinstance(element, chord.Chord):
        notes.append('.'.join(str(n) for n in element.normalOrder))

print("MIDI file processed successfully!")
print("Total notes/chords:", len(notes))
print("First 20:", notes[:20])

MIDI file processed successfully!
Total notes/chords: 1375
First 20: ['2.5.7.10', '2.5.7.10', '2.5.7.10', '2.5.7.10', '2.5.7.10', '4.7.10.0', '4.7.10.0', '2.5.7.10', '4.7.10.0', '2.5.7.10', '4.7.10.0', '4.7.10.0', '9.0.3.5', '9.0.3.5', '9.0.3.5', '7.10.0.3', '9.0.3.5', 'F3', 'F3', '10.2.5']


In [11]:
sequence_length = 10

sequences = []
next_notes = []

for i in range(len(notes) - sequence_length):
    sequences.append(notes[i:i + sequence_length])
    next_notes.append(notes[i + sequence_length])

print("Note sequences created successfully!")
print("Total sequences:", len(sequences))
print("Sequence length:", sequence_length)
print("First sequence:", sequences[0])
print("Next note:", next_notes[0])

Note sequences created successfully!
Total sequences: 1365
Sequence length: 10
First sequence: ['2.5.7.10', '2.5.7.10', '2.5.7.10', '2.5.7.10', '2.5.7.10', '4.7.10.0', '4.7.10.0', '2.5.7.10', '4.7.10.0', '2.5.7.10']
Next note: 4.7.10.0


In [12]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

encoder = LabelEncoder()

all_notes = notes
encoder.fit(all_notes)

X = np.array([
    [encoder.transform([n])[0] for n in seq]
    for seq in sequences
])

y = np.array([
    encoder.transform([n])[0]
    for n in next_notes
])

vocab_size = len(encoder.classes_)

print("Data converted successfully!")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of unique notes:", vocab_size)

Data converted successfully!
X shape: (1365, 10)
y shape: (1365,)
Number of unique notes: 157


In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential([
    Embedding(vocab_size, 64),
    LSTM(128),
    Dense(vocab_size, activation="softmax")
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam"
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
history = model.fit(
    X,
    y,
    epochs=20,
    batch_size=32,
    verbose=1
)

print("Training completed successfully! 🎵")

Epoch 1/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 4.7764
Epoch 2/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 4.3483
Epoch 3/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 4.2704
Epoch 4/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 4.1824
Epoch 5/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 4.0454
Epoch 6/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 3.9245
Epoch 7/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 3.8051
Epoch 8/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 3.6892
Epoch 9/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 3.5876
Epoch 10/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 3.4686
Epoch 11/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 3.3809
Epoch 12/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 3.2790
Epoch 13/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 3.1706
Epoch 14/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 3.0591
Epoch 15/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 2.9688
Epoc

In [15]:
import random

start_index = random.randint(0, len(X) - 1)
pattern = X[start_index].tolist()

generated_notes = []

for _ in range(50):
    input_sequence = np.array([pattern[-sequence_length:]])

    prediction = model.predict(input_sequence, verbose=0)
    predicted_index = np.argmax(prediction)

    generated_notes.append(encoder.inverse_transform([predicted_index])[0])
    pattern.append(predicted_index)

print("New music sequence generated successfully! 🎶")
print("Generated notes:")
print(generated_notes[:20])

New music sequence generated successfully! 🎶
Generated notes:
[np.str_('F4'), np.str_('F4'), np.str_('E-4'), np.str_('E-4'), np.str_('B-3'), np.str_('10.2.5'), np.str_('10.2.5'), np.str_('E-4'), np.str_('B-3'), np.str_('B-3'), np.str_('B-3'), np.str_('B-3'), np.str_('B-3'), np.str_('D4'), np.str_('B-3'), np.str_('B-3'), np.str_('B-3'), np.str_('B-3'), np.str_('B-3'), np.str_('B-3')]


In [16]:
from music21 import stream, note, chord

output_midi = stream.Stream()

for item in generated_notes:
    try:
        if "." in item:
            pitches = [int(p) for p in item.split(".")]
            output_midi.append(chord.Chord(pitches))
        else:
            output_midi.append(note.Note(item))
    except:
        continue

output_path = "AI_Generated_Music.mid"
output_midi.write("midi", fp=output_path)

print("🎵 MIDI file created successfully!")
print("Saved as:", output_path)

🎵 MIDI file created successfully!
Saved as: AI_Generated_Music.mid
